# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahd799/flyrank-internship-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
### Finding 1

The paper reports that content age is the strongest negative signal in its
growth-prediction model (logistic regression), while days visible and recent
impressions are among the strongest positive signals.

**My methodology question:** Where does the growth/decline label come from,
and is it based on a future outcome window that is kept separate from the
features used for prediction? The paper states that the ML pages are
exploratory appendix material and that health score (a related target
elsewhere in the paper) is partly constructed from some of the same inputs
used to explain it — so I would want to confirm the growth/decline label
here is genuinely independent of the predictor features, and that the
validation design supports the strength of the claim.

### Finding 2

The paper reports that the strongest measured freshness window is 31-90
days, with a 7.88:1 growth-to-decline ratio, and recommends refreshing
mature pages before they decay.

**My methodology question:** How is the growth-versus-decline label defined
for this comparison, and does the validation design account for differences
between clients and time periods? Since the paper itself describes this as
an observational study, I would treat this as a directional association
rather than evidence that updating a page directly causes growth.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Finding 1 source: ML Appendix - Growth & Classification (Logistic Regression)")
print("Finding 2 source: Finding #4 - The Freshness Multiplier")
print()
print("Both findings are labeled CONFIRMED in the paper and are backed by")
print("large sample sizes, but Finding 1 is explicitly exploratory ML")
print("appendix material, while Finding 2 is a direct aggregate comparison,")
print("which the paper treats as its primary evidence standard.")

Finding 1 source: ML Appendix - Growth & Classification (Logistic Regression)
Finding 2 source: Finding #4 - The Freshness Multiplier

Both findings are labeled CONFIRMED in the paper and are backed by
large sample sizes, but Finding 1 is explicitly exploratory ML
appendix material, while Finding 2 is a direct aggregate comparison,
which the paper treats as its primary evidence standard.


### Data verification

Before aggregating, I confirm the row counts and date ranges match the
expected partitions, and I grain-probe to confirm one row per
(report_date, client, content) with no duplicates.

In [11]:
# Data verification (per flyrank-data skill: always verify before aggregating)

# 1. Confirm row counts and date ranges match the partition we expect
print("March partition:")
print("  Rows:", len(march))
print("  Date range:", march["report_date"].min(), "to", march["report_date"].max())
print()
print("April partition:")
print("  Rows:", len(april))
print("  Date range:", april["report_date"].min(), "to", april["report_date"].max())
print()

# 2. Grain-probe: confirm one row per (report_date, client, content) — should be empty
grain_check_march = (
    march.groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="n")
)
duplicates = grain_check_march[grain_check_march["n"] > 1]
print("Duplicate grain rows in March (should be 0):", len(duplicates))

March partition:
  Rows: 9841378
  Date range: 2026-03-01 to 2026-03-31

April partition:
  Rows: 10424730
  Date range: 2026-04-01 to 2026-04-30

Duplicate grain rows in March (should be 0): 0


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Split design

I use a grouped split by client so that the same client cannot appear in
both training and test sets. This gives a more honest estimate of how the
model may perform on unseen clients.

To make the before/after comparison meaningful, I also run a naive random
split that ignores client grouping. This naive split represents what would
happen if leakage across a client's own pages were allowed between train
and test — the kind of split my model should NOT be evaluated with. I
compare naive vs. grouped Average Precision on the same data and features.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, confusion_matrix

# Load March and April data
march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

april = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet"
)

# Aggregate March data by client and page
march_agg = (
    march.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_sum_position=("gsc_sum_position", "sum")
    )
)

march_agg["march_position"] = (
    march_agg["march_sum_position"] / march_agg["march_impressions"]
)

# Aggregate April data
april_agg = (
    april.groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(april_impressions=("gsc_impressions", "sum"))
)

# Keep pages that had March impressions
model_df = march_agg[march_agg["march_impressions"] > 0].copy()

# Add April impressions
model_df = model_df.merge(
    april_agg,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)
model_df["april_impressions"] = model_df["april_impressions"].fillna(0)

# Target: impression decline from March to April
model_df["target_decline"] = (
    model_df["april_impressions"] < model_df["march_impressions"]
).astype(int)

features = ["march_impressions", "march_clicks", "march_position"]
target = "target_decline"

model_df = model_df.dropna(subset=features + [target])

print("Model rows:", len(model_df))
print("Target distribution:")
print(model_df[target].value_counts())

Model rows: 176738
Target distribution:
target_decline
1    111968
0     64770
Name: count, dtype: int64


### Population note

This analysis keeps only pages with march_impressions > 0. This filter is
based entirely on March data, which is before the label window, so it does
not leak outcome-window information — but it is a deliberate choice and is
disclosed here per the leakage-hunting checklist.

It also uses a single global March/April window for all clients rather than
per-client windows based on each client's gsc_data_start. Filtering on
march_impressions > 0 implicitly removes clients with no March activity, but
client history depth was not explicitly checked against
dim_clients.gsc_data_start, per the flyrank-data skill's panel warning.

In [12]:
# BEFORE: naive random split — pages from the same client can appear in
# both train and test, which risks an inflated / overly optimistic score
X = model_df[features].copy()
y = model_df[target].copy()

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.22, random_state=42, stratify=y
)

naive_model = DecisionTreeClassifier(
    max_depth=3, class_weight="balanced", random_state=42
)
naive_model.fit(X_train_naive, y_train_naive)
naive_prob = naive_model.predict_proba(X_test_naive)[:, 1]
naive_ap = average_precision_score(y_test_naive, naive_prob)

print("Naive random-split Average Precision:", naive_ap)

Naive random-split Average Precision: 0.6739798771774459


In [13]:
# AFTER: honest grouped split — clients in train and test are completely separate
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
test_clients = set(model_df.iloc[test_idx]["client_hash_id"])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Shared clients:", len(train_clients & test_clients))

model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]
honest_ap = average_precision_score(y_test, test_prob)

print("Honest grouped-split Average Precision:", honest_ap)
test_prob = model.predict_proba(X_test)[:, 1]

honest_ap = average_precision_score(y_test, test_prob)

print("Honest grouped-split Average Precision:", honest_ap)

Train rows: 133491
Test rows: 43247
Train clients: 36
Test clients: 11
Shared clients: 0
Honest grouped-split Average Precision: 0.6113944595078553
Honest grouped-split Average Precision: 0.6113944595078553


### Base rate check

Every score should sit next to its naive baseline. I print the base rate
(share of the positive class) so the Average Precision numbers above can be
read relative to what a skill-less model would achieve.

In [14]:
# Base rate — the naive "always predict the majority class" baseline
base_rate = y_test.mean()
print("Base rate (share of positive class in test set):", round(base_rate, 4))
print("A model with no real skill would score Average Precision near this base rate.")
print(f"Honest grouped-split AP ({honest_ap:.4f}) vs base rate ({base_rate:.4f})")
print(f"Skill above base rate: {honest_ap - base_rate:.4f}")

Base rate (share of positive class in test set): 0.5776
A model with no real skill would score Average Precision near this base rate.
Honest grouped-split AP (0.6114) vs base rate (0.5776)
Skill above base rate: 0.0338


In [15]:
comparison = pd.DataFrame({
    "Method": [
        "Naive random split (client leakage risk)",
        "Honest grouped split (by client)"
    ],
    "Average Precision": [naive_ap, honest_ap]
})

comparison

,Method,Average Precision
0,Naive random split (client leakage risk),0.673980
1,Honest grouped split (by client),0.611394


The naive random split and the honest grouped split produce different
Average Precision scores on the same data and features. This gap is an
observed indicator of client-level leakage risk: when pages from the same
client can appear in both train and test, the measured score can be more
optimistic than what the model would actually achieve on entirely new,
unseen clients. The grouped split is treated as the more honest estimate
going forward.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit on the final feature set

I check three things: (1) whether any feature encodes information from
outside the March window, (2) whether any (client, page) row leaks across
the train/test boundary, and (3) whether any feature is suspiciously
correlated with the target in a way that would suggest it is a disguised
proxy for the label.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3a. Feature source check
print("Features:", features)
print("- march_impressions, march_clicks: summed directly from March GSC data")
print("- march_position: weighted average computed from March GSC data")
print("None of these reference april_impressions, so there is no direct")
print("future-window leak in the feature set.")
print()

# 3b. Row-level overlap check between train and test
train_keys = set(zip(
    model_df.iloc[train_idx]["client_hash_id"],
    model_df.iloc[train_idx]["content_hash_id"]
))
test_keys = set(zip(
    model_df.iloc[test_idx]["client_hash_id"],
    model_df.iloc[test_idx]["content_hash_id"]
))
print("Overlapping (client, page) rows between train and test:", len(train_keys & test_keys))
print()

# 3c. Correlation sanity check
corr_check = model_df[features + [target]].corr()[target].drop(target)
print("Feature correlation with target:")
print(corr_check)

Features: ['march_impressions', 'march_clicks', 'march_position']
- march_impressions, march_clicks: summed directly from March GSC data
- march_position: weighted average computed from March GSC data
None of these reference april_impressions, so there is no direct
future-window leak in the feature set.

Overlapping (client, page) rows between train and test: 0

Feature correlation with target:
march_impressions    0.007726
march_clicks        -0.035604
march_position      -0.032048
Name: target_decline, dtype: float64


All three features are aggregated from March data only, with no reference
to April values, so there is no direct future-window leak. There is zero
row-level overlap between train and test after the grouped split. No
feature shows a correlation with the target near ±1.0, which would be the
signature of a feature that is effectively a restatement of the label.

Conclusion: no leakage was found in this feature set under this split
design. As the paper itself notes when discussing its own Random Forest
health-score model, correlation and feature importance are descriptive,
not causal — this audit checks for leakage, not for "why" a feature matters.

### Verifying the leakage-detection harness

To confirm the evaluation setup itself is capable of detecting leakage, I
deliberately inject a known-leaky feature (april_impressions, which directly
determines the label by construction) and check that the score jumps toward
1.0. If it didn't, that would mean the harness is not sensitive to leakage,
and the "no leakage found" conclusion above could not be trusted.

In [17]:
# Verify the leakage-detection harness itself works:
# deliberately inject a leaky feature (april_impressions) and confirm the
# score jumps toward 1.0 — if it doesn't, the test setup itself is broken.
# NOTE: max_depth is intentionally unrestricted here (unlike the main model)
# because this diagnostic tree needs enough capacity to actually exploit
# the injected leak — a shallow tree may under-use it and hide the leakage.
leaky_features = features + ["april_impressions"]
X_leaky = model_df[leaky_features].copy()

X_train_leak = X_leaky.iloc[train_idx]
X_test_leak = X_leaky.iloc[test_idx]

leaky_model = DecisionTreeClassifier(class_weight="balanced", random_state=42)
leaky_model.fit(X_train_leak, y_train)
leaky_prob = leaky_model.predict_proba(X_test_leak)[:, 1]
leaky_ap = average_precision_score(y_test, leaky_prob)

print("Honest AP (no leak):", honest_ap)
print("AP WITH deliberate leak (april_impressions included, unrestricted depth):", leaky_ap)

Honest AP (no leak): 0.6113944595078553
AP WITH deliberate leak (april_impressions included, unrestricted depth): 0.9909232942475172


Adding april_impressions as a feature — which directly determines the label
by construction — pushes Average Precision sharply toward 1.0. This confirms
the evaluation harness is working correctly: it does detect leakage when
leakage is deliberately introduced, which increases confidence that the
"no leakage found" conclusion for the real feature set is a genuine finding
and not a blind spot in the test itself.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Rewriting my boldest claim

**Original claim (Week-5):** "The model is therefore relying mainly on
existing search engagement and visibility signals."

This overstates certainty — it implies a general, causal explanation of
model behavior rather than a result specific to one model, one split, and
one dataset.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(feature_importance)

print()
print("Rewritten claim (safe language):")
print("Based on the observed feature importance in this grouped-split")
print("evaluation, march_impressions shows the largest measured contribution")
print("to the tree's splits, followed by march_position and march_clicks.")
print("This is a directional signal from one model on one split, not a")
print("general claim about what causes decline -- and the model's output")
print("should be used as decision-support, not as a certain prediction.")

             feature  importance
0  march_impressions    0.420269
2     march_position    0.306840
1       march_clicks    0.272891

Rewritten claim (safe language):
Based on the observed feature importance in this grouped-split
evaluation, march_impressions shows the largest measured contribution
to the tree's splits, followed by march_position and march_clicks.
This is a directional signal from one model on one split, not a
general claim about what causes decline -- and the model's output
should be used as decision-support, not as a certain prediction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.